In [1]:
import math
import os
import pickle

from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

In [2]:
output_path = "../../output/protenn2/v5"
input_data_folder = "../../datasets/v5"
model_path = os.path.join(output_path, "best_model.pt")
label_encoder_path = os.path.join(output_path, "label_encoder.pkl")

In [3]:
import colorsys
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt
import os
import torch


# --- Helper functions for custom CATH coloring ---

def parse_cath(cath_string):
    """
    Parses a CATH string (e.g., "3.30.559.10") into its components.
    Returns (Class, Architecture, Topology, Homologous Superfamily) as strings.
    If a component is missing, it returns an empty string for that part.
    Handles 'NO_DOMAIN_REGION' specifically.
    """
    if cath_string == "NO_DOMAIN_REGION":
        return None, None, None, None
    parts = cath_string.split('.')
    # Pad with empty strings if parts are missing, to always return 4 components
    return (parts + [''] * 4)[:4]


def get_cath_color(cath_label_string, plot_random_seed=0):
    """
    Generates an RGBA color for a CATH domain label, optimizing for differentiability
    within a specific plot, while keeping base class color consistent across plots.

    Args:
        cath_label_string (str): The CATH domain string or "NO_DOMAIN_REGION".
        plot_random_seed (int): A unique seed for the current plot to randomize shades.
                                Ensures different shade patterns per visualization.
    """
    if cath_label_string == "NO_DOMAIN_REGION":
        # Keep NO_DOMAIN_REGION color consistent and easily recognizable
        return mcolors.to_rgba('lightgray', alpha=0.7)

    class_id, arch_id, topo_id, hs_id = parse_cath(cath_label_string)

    # Define base hues for CATH Classes (H from HSV, 0.0-1.0 range)
    # These are fixed and widely separated for consistent inter-class distinction.
    class_hues = {
        '1': 0.0,  # Alpha (Pure Red)
        '2': 0.66,  # Beta (Pure Blue)
        '3': 0.33,  # Alpha-Beta (Pure Green)
        '4': 0.15,  # Few Secondary Structures (Orange/Yellow)
        None: 0.5  # Default hue for unparsed/unexpected class_id
    }

    base_hue = class_hues.get(class_id, 0.5)

    # --- Aggressive modulation for Architecture and Topology based on CATH ID + plot_random_seed ---

    # Combine CATH ID parts and the plot_random_seed for unique seeds per label per plot
    base_id_hash = hash(cath_label_string) if cath_label_string else 0

    # 1. Hue Modulation:
    #   Apply a significant hue shift for strong differentiation *within* the current plot's class.
    #   The 'plot_random_seed' ensures the pattern of shifts changes per image.
    hue_mod_seed = base_id_hash + plot_random_seed * 113  # Multiply by prime to mix it well
    hue_shift_magnitude = 0.2  # Max shift of 20% of the color wheel
    hue_shift = (hash(hue_mod_seed) % 1000) / 1000.0 * hue_shift_magnitude - (hue_shift_magnitude / 2)

    hue = (base_hue + hue_shift) % 1.0  # Ensure hue wraps around 0-1

    # 2. Saturation Modulation:
    #   Maximize the range of saturation.
    saturation_mod_seed = base_id_hash + plot_random_seed * 137
    saturation = 0.2 + (
            (hash(saturation_mod_seed) % 1000) / 1000.0 * 0.8)  # Range from 0.2 (very pale) to 1.0 (full vibrant)

    # 3. Value/Lightness Modulation:
    #   Maximize the range of lightness.
    value_mod_seed = base_id_hash + plot_random_seed * 151
    value = 0.3 + ((hash(value_mod_seed) % 1000) / 1000.0 * 0.7)  # Range from 0.3 (quite dark) to 1.0 (full bright)

    # Ensure HSV values are within [0, 1] bounds after modulation
    saturation = np.clip(saturation, 0.0, 1.0)
    value = np.clip(value, 0.0, 1.0)

    # Convert HSV to RGB
    r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
    return (r, g, b, 1.0)  # Return with full opacity (alpha=1.0)


# --- Modified visualize_predictions function ---

def visualize_predictions(model, dataloader, label_encoder, output_dir, device, num_visualizations=5):
    """
    Visualizes per-residue predictions for a subset of the validation set,
    showing the *entire padded protein sequence*, with a new color pattern for each image
    while keeping base class color (C in CATH) consistent.

    Args:
        model (torch.nn.Module): The trained model.
        dataloader (DataLoader): DataLoader for the validation set.
        label_encoder (LabelEncoder): The LabelEncoder used during training.
        output_dir (str): Directory to save the visualization plots.
        device (torch.device): The device (CPU or MPS) to run inference on.
        num_visualizations (int): Number of proteins to visualize.
    """
    model.eval()
    os.makedirs(output_dir, exist_ok=True)

    # Reverse mapping from encoded ID to label string
    id_to_label = {i: label for i, label in enumerate(label_encoder.classes_)}

    visualized_count = 0

    with torch.no_grad():
        for i, (x, y_true, domain_id) in enumerate(dataloader):
            if visualized_count >= num_visualizations:
                break

            # Handle domain_id (assuming batch_size=1, domain_id is typically a list/tuple)
            current_domain_id_str = domain_id[0] if isinstance(domain_id, (list, tuple, torch.Tensor)) else str(
                domain_id)

            # --- Generate a unique random seed for THIS specific plot ---
            # Using hash of the domain ID ensures it's consistent for the same protein
            # but different for different proteins.
            plot_specific_seed = hash(current_domain_id_str)
            # You could also use a simple counter: plot_specific_seed = visualized_count
            # Or a true random number: plot_specific_seed = np.random.randint(0, 1000000)
            # but hashing the domain ID is better for reproducibility if you re-run for same protein.

            for k, v in x.items():
                x[k] = v.to(device, non_blocking=True)
            y_true = y_true.to(device, non_blocking=True)

            outputs = model(x)
            outputs = outputs.permute(0, 2, 1)
            y_pred = torch.argmax(outputs, dim=1)

            for batch_idx in range(x["embedding"].shape[0]):
                if visualized_count >= num_visualizations:
                    break

                full_display_length = x["embedding"].shape[1]

                true_labels_tensor = y_true[batch_idx].cpu()
                pred_labels_tensor = y_pred[batch_idx].cpu()

                true_labels_to_plot = true_labels_tensor.numpy()
                pred_labels_to_plot = pred_labels_tensor.numpy()

                true_labels_str = [id_to_label[int(id_val)] for id_val in true_labels_to_plot]
                pred_labels_str = [id_to_label[int(id_val)] for id_val in pred_labels_to_plot]

                # --- Custom Coloring Logic ---
                all_labels_in_current_plot = sorted(list(set(true_labels_str + pred_labels_str)))
                label_to_plot_value = {label: idx for idx, label in enumerate(all_labels_in_current_plot)}

                # IMPORTANT: Pass the plot_specific_seed to get_cath_color
                label_to_rgba_color = {label: get_cath_color(label, plot_random_seed=plot_specific_seed)
                                       for label in all_labels_in_current_plot}

                cmap_colors_list = [label_to_rgba_color[label] for label in all_labels_in_current_plot]
                custom_cmap = mcolors.ListedColormap(cmap_colors_list)

                true_plot_values = [label_to_plot_value[label] for label in true_labels_str]
                pred_plot_values = [label_to_plot_value[label] for label in pred_labels_str]

                vmin_plot = 0
                vmax_plot = len(all_labels_in_current_plot) - 1
                if vmax_plot == 0: vmax_plot = 1

                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
                fig.suptitle(f"Domain {current_domain_id_str} Residue-wise Prediction (Full Sequence)")

                ax1.imshow(np.array(true_plot_values).reshape(1, -1), cmap=custom_cmap, aspect='auto',
                           extent=[0, full_display_length, 0, 1],
                           vmin=vmin_plot, vmax=vmax_plot)
                ax1.set_yticks([])
                ax1.set_title("True CATH Domains")
                ax1.set_ylabel("True")
                ax1.set_xlim(0, full_display_length)

                ax2.imshow(np.array(pred_plot_values).reshape(1, -1), cmap=custom_cmap, aspect='auto',
                           extent=[0, full_display_length, 0, 1],
                           vmin=vmin_plot, vmax=vmax_plot)
                ax2.set_yticks([])
                ax2.set_title("Predicted CATH Domains")
                ax2.set_xlabel("Residue Index")
                ax2.set_ylabel("Predicted")
                ax2.set_xlim(0, full_display_length)

                handles = [plt.Rectangle((0, 0), 1, 1, color=label_to_rgba_color[label])
                           for label in all_labels_in_current_plot]
                ax2.legend(handles, all_labels_in_current_plot, loc='upper center', bbox_to_anchor=(0.5, -0.2),
                           fancybox=True, shadow=True, ncol=3)

                plt.tight_layout(rect=[0, 0.03, 1, 0.95])
                plt.savefig(os.path.join(output_dir, f"{current_domain_id_str}.png"))
                plt.close(fig)

                visualized_count += 1
                print(f"Generated visualization for protein {visualized_count}")

In [4]:
from src.protenn2.model import CathPredEnn2
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.utils import get_train_val_test_paths, calculate_max_protein_length

# Define paths (adjust these to your specific project structure)
# IMPORTANT: Make sure these paths point to where your trained model,
# label encoder, and dataset CSVs/embeddings are located.


visualization_output_dir = os.path.join(output_path, "visualization_results")

# Set device (MPS for Apple Silicon, otherwise CPU)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU) for visualization.")
else:
    device = torch.device("cpu")
    print("MPS not available, falling back to CPU for visualization.")

# Load LabelEncoder
try:
    with open(label_encoder_path, "rb") as f:
        label_encoder = pickle.load(f)
    print(f"Loaded LabelEncoder with {len(label_encoder.classes_)} classes.")
except FileNotFoundError:
    print(f"Error: LabelEncoder file not found at {label_encoder_path}. Please check the path.")
    exit()  # Exit if the label encoder isn't found

# Get data paths
train_path, val_path, test_path = get_train_val_test_paths(input_data_folder)

# Create validation dataset and dataloader
# Pass the correct embedding_dir to the dataset constructor
val_dataset = CathPredPerResidueDataset(val_path, label_encoder,
                                        embedding_dir="../../data/embeddings/protein_embeddings_new", fit=False)
max_protein_length = calculate_max_protein_length(input_data_folder)

if max_protein_length == 0:
    print(
        "Error: Max protein length is 0. This usually means no valid protein embeddings were found or data paths are incorrect.")
    print(
        "Please ensure 'input_data_folder' and 'embedding_base_dir' are correctly set and contain the necessary files.")
    exit()

collate_fn = create_protein_collate_fn(max_protein_length, val_dataset.no_domain_encoded_id)
# Use batch_size=1 for easier visualization of individual proteins
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

# Initiate and load the model
num_classes = len(label_encoder.classes_)
model = CathPredEnn2(num_classes=num_classes)
try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    print("Model loaded successfully.")
except FileNotFoundError:
    print(f"Error: Model file not found at {model_path}. Please check the path.")
    exit()
except Exception as e:
    print(f"Error loading model: {e}. Ensure CathPredEnn2 model definition matches the saved state_dict.")
    exit()

# Perform and visualize predictions
print("\nStarting visualization...")
visualize_predictions(model, val_dataloader, label_encoder, visualization_output_dir, device,
                      num_visualizations=100)
print(f"\nVisualizations saved to {visualization_output_dir}")

Using MPS (Apple Silicon GPU) for visualization.
Loaded LabelEncoder with 879 classes.
Dataset initialized with 1319 unique proteins.
Max protein length: 599
Model loaded successfully.

Starting visualization...
Generated visualization for protein 1
Generated visualization for protein 2
Generated visualization for protein 3
Generated visualization for protein 4
Generated visualization for protein 5
Generated visualization for protein 6
Generated visualization for protein 7
Generated visualization for protein 8
Generated visualization for protein 9
Generated visualization for protein 10
Generated visualization for protein 11
Generated visualization for protein 12
Generated visualization for protein 13
Generated visualization for protein 14
Generated visualization for protein 15
Generated visualization for protein 16
Generated visualization for protein 17
Generated visualization for protein 18
Generated visualization for protein 19
Generated visualization for protein 20
Generated visuali